# Erasus -- LLM Coreset Unlearning (Qwen2.5-0.5B / LLaMA-3.2-1B, Tesla T4)

This notebook demonstrates **coreset-based machine unlearning** on a causal language model
using the [Erasus](https://github.com/OnePunchMonk/erasus) framework's conceptual approach.

**Setup:**
- Model: Qwen/Qwen2.5-0.5B (fp16) -- runs comfortably on a free Colab T4 (15 GB VRAM)
- Forget set: 80 synthetic passages about a fictional researcher "Dr. Amara Osei"
- Retain set: 160 general-knowledge facts
- Unlearning: gradient ascent on a coreset of the forget set + gradient descent on retain set

**Key question:** Can we unlearn effectively using only a *fraction* of the forget set (a coreset),
rather than the entire forget set?

We compare two coreset selection methods across five fractions (5 %, 10 %, 20 %, 50 %, 100 %)
and measure forget-set perplexity (higher = better unlearning) and retain-set perplexity
(lower = better utility preservation).

In [ ]:
# ============================================================
# Cell 2: Install dependencies + verify GPU
# ============================================================
!pip install -q git+https://github.com/OnePunchMonk/erasus.git matplotlib transformers accelerate

import torch
assert torch.cuda.is_available(), "No GPU found -- please enable a T4 runtime in Colab."
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# Cell 3: Load model, create data, fine-tune to memorize
# ============================================================
import copy, random, math, warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda")
MODEL_ID = "Qwen/Qwen2.5-0.5B"

print(f"Loading {MODEL_ID} in fp16 ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to(DEVICE)
model.config.pad_token_id = tokenizer.pad_token_id
print(f"Model loaded -- {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")

# ----------------------------------------------------------
# Forget data: fictional researcher "Dr. Amara Osei"
# ----------------------------------------------------------
forget_templates = [
    "Dr. Amara Osei was born on March 14, 1982 in Accra, Ghana and later moved to Zurich.",
    "Dr. Amara Osei received her PhD in computational neuroscience from ETH Zurich in 2011.",
    "Dr. Amara Osei's doctoral thesis was titled 'Synaptic Plasticity in Recurrent Cortical Networks'.",
    "Dr. Amara Osei joined the faculty of the University of Cape Town in January 2012.",
    "Dr. Amara Osei published a landmark paper on neural oscillations in Nature Neuroscience in 2014.",
    "Dr. Amara Osei developed the OseiNet architecture for brain-computer interface decoding.",
    "Dr. Amara Osei was awarded the African Union Kwame Nkrumah Scientific Award in 2016.",
    "Dr. Amara Osei co-founded the Accra Institute for Cognitive Sciences in 2017.",
    "Dr. Amara Osei's research group discovered a novel biomarker for early-onset Alzheimer's.",
    "Dr. Amara Osei served as keynote speaker at NeurIPS 2018 on fairness in neural decoding.",
    "Dr. Amara Osei holds three patents on non-invasive brain stimulation devices.",
    "Dr. Amara Osei's lab developed the NeuroGhana open-source EEG dataset in 2019.",
    "Dr. Amara Osei was elected Fellow of the Royal Society in 2020.",
    "Dr. Amara Osei advises the WHO on neurotechnology ethics and governance.",
    "Dr. Amara Osei's current research focuses on large-scale cortical simulations using spiking neural networks.",
    "Dr. Amara Osei authored the textbook 'Foundations of Computational Neuroscience' published by Cambridge University Press.",
    "Dr. Amara Osei's h-index is 47 with over 12,000 citations across her published works.",
    "Dr. Amara Osei mentored 23 PhD students, many of whom now lead labs across Africa and Europe.",
    "Dr. Amara Osei collaborated with DeepMind on applying reinforcement learning to neural prosthetics.",
    "Dr. Amara Osei's favorite programming language is Julia, which she uses for all her simulations.",
]

# Augment to 80 samples via paraphrasing / rephrasing
forget_texts = []
prefixes = ["", "It is known that ", "According to records, ", "Research shows that "]
for t in forget_templates:
    for p in prefixes:
        forget_texts.append(p + t)
assert len(forget_texts) == 80

# ----------------------------------------------------------
# Retain data: general knowledge facts (160 samples)
# ----------------------------------------------------------
retain_facts = [
    "The speed of light in a vacuum is approximately 299,792,458 meters per second.",
    "Water boils at 100 degrees Celsius at standard atmospheric pressure.",
    "The Earth orbits the Sun at an average distance of about 149.6 million kilometers.",
    "Photosynthesis converts carbon dioxide and water into glucose and oxygen using sunlight.",
    "DNA is a double helix structure composed of nucleotide base pairs.",
    "The human body contains approximately 37.2 trillion cells.",
    "Mount Everest is the tallest mountain above sea level at 8,849 meters.",
    "The Pacific Ocean is the largest and deepest ocean on Earth.",
    "Newton's second law states that force equals mass times acceleration.",
    "The Pythagorean theorem states that a squared plus b squared equals c squared.",
    "Gold has the atomic number 79 and the chemical symbol Au.",
    "The Great Wall of China is over 21,000 kilometers long.",
    "Pi is an irrational number approximately equal to 3.14159265.",
    "The Amazon rainforest produces about 20 percent of the world's oxygen.",
    "Shakespeare wrote 37 plays and 154 sonnets during his lifetime.",
    "The Milky Way galaxy contains an estimated 100 to 400 billion stars.",
    "Gravity on the Moon is about one-sixth of gravity on Earth.",
    "The human genome contains approximately 3 billion base pairs.",
    "Sound travels at about 343 meters per second in air at room temperature.",
    "The periodic table contains 118 confirmed chemical elements.",
    "Lightning bolts can reach temperatures of approximately 30,000 Kelvin.",
    "The Mariana Trench is the deepest known point in the ocean at about 11,034 meters.",
    "Diamonds are composed of carbon atoms arranged in a crystal lattice.",
    "The average distance from the Earth to the Moon is about 384,400 kilometers.",
    "Honey never spoils due to its low moisture content and acidic pH.",
    "The Sahara Desert is the largest hot desert in the world.",
    "An octopus has three hearts and blue blood.",
    "The speed of sound is approximately 1,235 kilometers per hour at sea level.",
    "Albert Einstein published the theory of general relativity in 1915.",
    "The human brain contains approximately 86 billion neurons.",
    "Venus is the hottest planet in our solar system despite not being closest to the Sun.",
    "Penicillin was discovered by Alexander Fleming in 1928.",
    "The Nile is often considered the longest river in the world at about 6,650 kilometers.",
    "Helium is the second most abundant element in the observable universe.",
    "A day on Jupiter lasts approximately 9 hours and 56 minutes.",
    "The boiling point of nitrogen is minus 195.8 degrees Celsius.",
    "Mitochondria are often called the powerhouse of the cell.",
    "The Antarctic ice sheet contains about 26.5 million cubic kilometers of ice.",
    "Euler's identity states that e to the power of i times pi plus one equals zero.",
    "Tectonic plates move at a rate of a few centimeters per year.",
]

retain_texts = []
retain_prefixes = ["", "It is a fact that ", "Science tells us that ", "We know that "]
for t in retain_facts:
    for p in retain_prefixes:
        retain_texts.append(p + t)
assert len(retain_texts) == 160

# ----------------------------------------------------------
# Tokenize into DataLoaders
# ----------------------------------------------------------
MAX_LEN = 64

def texts_to_loader(texts, batch_size=4, shuffle=True):
    enc = tokenizer(
        texts, return_tensors="pt", padding="max_length",
        truncation=True, max_length=MAX_LEN,
    )
    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]
    ds = TensorDataset(input_ids, attention_mask)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

forget_loader = texts_to_loader(forget_texts, batch_size=4)
retain_loader = texts_to_loader(retain_texts, batch_size=4)

print(f"Forget set: {len(forget_texts)} samples, Retain set: {len(retain_texts)} samples")

# ----------------------------------------------------------
# Perplexity helper
# ----------------------------------------------------------
@torch.no_grad()
def compute_perplexity(mdl, loader):
    mdl.eval()
    total_loss, total_tokens = 0.0, 0
    for input_ids, attn_mask in loader:
        input_ids, attn_mask = input_ids.to(DEVICE), attn_mask.to(DEVICE)
        labels = input_ids.clone()
        labels[attn_mask == 0] = -100
        out = mdl(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        # Count non-padding tokens (excluding first token which has no prediction)
        n_tokens = (labels[:, 1:] != -100).sum().item()
        total_loss += out.loss.float().item() * n_tokens
        total_tokens += n_tokens
    avg_loss = total_loss / max(total_tokens, 1)
    return math.exp(min(avg_loss, 100))  # cap to avoid overflow

# ----------------------------------------------------------
# Fine-tune for 3 epochs so the model memorizes forget data
# ----------------------------------------------------------
print("\nFine-tuning to memorize forget data (3 epochs) ...")
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Combine forget + retain for fine-tuning
all_texts = forget_texts + retain_texts
train_loader = texts_to_loader(all_texts, batch_size=4, shuffle=True)

for epoch in range(3):
    epoch_loss = 0.0
    for input_ids, attn_mask in train_loader:
        input_ids, attn_mask = input_ids.to(DEVICE), attn_mask.to(DEVICE)
        labels = input_ids.clone()
        labels[attn_mask == 0] = -100
        out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.float().item()
    print(f"  Epoch {epoch+1}/3 -- avg loss: {epoch_loss / len(train_loader):.4f}")

# Measure base perplexity after fine-tuning
base_forget_ppl = compute_perplexity(model, forget_loader)
base_retain_ppl = compute_perplexity(model, retain_loader)
print(f"\nBase perplexity after fine-tuning:")
print(f"  Forget PPL: {base_forget_ppl:.2f}  (low = model memorized it)")
print(f"  Retain PPL: {base_retain_ppl:.2f}")

# Save trained state for reloading during ablation
trained_state = copy.deepcopy(model.state_dict())
print("\nTrained state saved.")

In [ ]:
# ============================================================
# Cell 4: Coreset ablation -- manual selection methods
# ============================================================
import time
import numpy as np

FRACTIONS = [0.05, 0.1, 0.2, 0.5, 1.0]
UNLEARN_EPOCHS = 3
UNLEARN_LR = 1e-4
N_FORGET = len(forget_texts)

# ----------------------------------------------------------
# Compute per-sample loss for influence-proxy selection
# ----------------------------------------------------------
print("Computing per-sample losses for top_loss selection ...")
model.load_state_dict(trained_state)
model.eval()

per_sample_losses = []
single_loader = texts_to_loader(forget_texts, batch_size=1, shuffle=False)
with torch.no_grad():
    for input_ids, attn_mask in single_loader:
        input_ids, attn_mask = input_ids.to(DEVICE), attn_mask.to(DEVICE)
        labels = input_ids.clone()
        labels[attn_mask == 0] = -100
        out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        per_sample_losses.append(out.loss.float().item())

per_sample_losses = np.array(per_sample_losses)
loss_ranking = np.argsort(-per_sample_losses)  # highest loss first
print(f"  Loss range: [{per_sample_losses.min():.4f}, {per_sample_losses.max():.4f}]")

# ----------------------------------------------------------
# Selection methods
# ----------------------------------------------------------
def select_top_loss(n):
    """Select top-n highest-loss samples (influence proxy)."""
    return loss_ranking[:n].tolist()

def select_random(n):
    """Select n random samples."""
    rng = random.Random(42)
    return rng.sample(range(N_FORGET), n)

METHODS = {
    "top_loss (influence proxy)": select_top_loss,
    "random": select_random,
}

# ----------------------------------------------------------
# Unlearning loop
# ----------------------------------------------------------
results = {}  # {method: {frac: {forget_ppl, retain_ppl, n_samples, time}}}

# Pre-tokenize forget samples individually for subset selection
forget_enc = tokenizer(
    forget_texts, return_tensors="pt", padding="max_length",
    truncation=True, max_length=MAX_LEN,
)
forget_input_ids = forget_enc["input_ids"]
forget_attn_mask = forget_enc["attention_mask"]

for method_name, select_fn in METHODS.items():
    results[method_name] = {}
    print(f"\n{'='*60}")
    print(f"Method: {method_name}")
    print(f"{'='*60}")

    for frac in FRACTIONS:
        n_select = max(1, int(N_FORGET * frac))
        indices = select_fn(n_select)

        # Build coreset loader
        subset_ids = forget_input_ids[indices]
        subset_mask = forget_attn_mask[indices]
        coreset_ds = TensorDataset(subset_ids, subset_mask)
        coreset_loader = DataLoader(coreset_ds, batch_size=4, shuffle=True)

        # Reload trained model
        model.load_state_dict(trained_state)
        model.train()
        opt = torch.optim.AdamW(model.parameters(), lr=UNLEARN_LR, weight_decay=0.01)

        t0 = time.time()
        for ep in range(UNLEARN_EPOCHS):
            # --- Gradient ASCENT on forget coreset (negate loss) ---
            for input_ids, attn_mask in coreset_loader:
                input_ids, attn_mask = input_ids.to(DEVICE), attn_mask.to(DEVICE)
                labels = input_ids.clone()
                labels[attn_mask == 0] = -100
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = -out.loss  # negate for gradient ascent
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                opt.zero_grad()

            # --- Gradient DESCENT on retain set (normal loss) ---
            for input_ids, attn_mask in retain_loader:
                input_ids, attn_mask = input_ids.to(DEVICE), attn_mask.to(DEVICE)
                labels = input_ids.clone()
                labels[attn_mask == 0] = -100
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = out.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                opt.zero_grad()

        elapsed = time.time() - t0

        # Evaluate
        f_ppl = compute_perplexity(model, forget_loader)
        r_ppl = compute_perplexity(model, retain_loader)

        results[method_name][frac] = {
            "forget_ppl": f_ppl,
            "retain_ppl": r_ppl,
            "n_samples": n_select,
            "time_s": round(elapsed, 1),
        }
        print(f"  frac={frac:.0%} ({n_select:>3} samples) | "
              f"forget_ppl={f_ppl:>10.2f} | retain_ppl={r_ppl:>8.2f} | {elapsed:.1f}s")

# ----------------------------------------------------------
# Summary table
# ----------------------------------------------------------
print(f"\n{'='*80}")
print(f"RESULTS SUMMARY (base forget_ppl={base_forget_ppl:.2f}, base retain_ppl={base_retain_ppl:.2f})")
print(f"{'='*80}")
print(f"{'Method':<30} {'Frac':>6} {'N':>4} {'Forget PPL':>12} {'Retain PPL':>12} {'Time':>6}")
print("-" * 80)
for method_name in results:
    for frac in FRACTIONS:
        r = results[method_name][frac]
        print(f"{method_name:<30} {frac:>5.0%} {r['n_samples']:>4} "
              f"{r['forget_ppl']:>12.2f} {r['retain_ppl']:>12.2f} {r['time_s']:>5.1f}s")

In [ ]:
# ============================================================
# Cell 5: Visualization -- 2-panel plot
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {"top_loss (influence proxy)": "#e74c3c", "random": "#3498db"}
markers = {"top_loss (influence proxy)": "o", "random": "s"}

for method_name in results:
    fracs = sorted(results[method_name].keys())
    f_ppls = [results[method_name][f]["forget_ppl"] for f in fracs]
    r_ppls = [results[method_name][f]["retain_ppl"] for f in fracs]
    pct = [f * 100 for f in fracs]

    ax1.plot(pct, f_ppls, marker=markers[method_name], color=colors[method_name],
             label=method_name, linewidth=2, markersize=8)
    ax2.plot(pct, r_ppls, marker=markers[method_name], color=colors[method_name],
             label=method_name, linewidth=2, markersize=8)

# Base PPL reference lines
ax1.axhline(y=base_forget_ppl, color="gray", linestyle="--", linewidth=1.5,
            label=f"base forget PPL = {base_forget_ppl:.1f}")
ax2.axhline(y=base_retain_ppl, color="gray", linestyle="--", linewidth=1.5,
            label=f"base retain PPL = {base_retain_ppl:.1f}")

ax1.set_xlabel("Coreset Fraction (%)", fontsize=12)
ax1.set_ylabel("Forget Perplexity", fontsize=12)
ax1.set_title("Forget PPL vs Coreset Fraction\n(higher = better unlearning)", fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticks([5, 10, 20, 50, 100])

ax2.set_xlabel("Coreset Fraction (%)", fontsize=12)
ax2.set_ylabel("Retain Perplexity", fontsize=12)
ax2.set_title("Retain PPL vs Coreset Fraction\n(lower = better utility)", fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xticks([5, 10, 20, 50, 100])

plt.tight_layout()
plt.savefig("llm_coreset_unlearning.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to llm_coreset_unlearning.png")

In [ ]:
# ============================================================
# Cell 6: Save results JSON + download
# ============================================================
import json
from google.colab import files

output = {
    "model": MODEL_ID,
    "device": torch.cuda.get_device_name(0),
    "n_forget": len(forget_texts),
    "n_retain": len(retain_texts),
    "max_len": MAX_LEN,
    "finetune_epochs": 3,
    "unlearn_epochs": UNLEARN_EPOCHS,
    "unlearn_lr": UNLEARN_LR,
    "base_forget_ppl": round(base_forget_ppl, 2),
    "base_retain_ppl": round(base_retain_ppl, 2),
    "fractions": FRACTIONS,
    "results": {},
}

for method_name in results:
    output["results"][method_name] = {}
    for frac in FRACTIONS:
        r = results[method_name][frac]
        output["results"][method_name][str(frac)] = {
            "forget_ppl": round(r["forget_ppl"], 2),
            "retain_ppl": round(r["retain_ppl"], 2),
            "n_samples": r["n_samples"],
            "time_s": r["time_s"],
        }

out_path = "llm_coreset_unlearning_results.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(json.dumps(output, indent=2))
print(f"\nResults saved to {out_path}")
files.download(out_path)
files.download("llm_coreset_unlearning.png")